<a href="https://colab.research.google.com/github/lolkamivon/Genetic-Chess-Engine/blob/main/chess_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 50.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=2ab9d26e1b9433b2a45a6d27e4a786558404381f15db87f8b5de79a7ab2bc805
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


In [10]:
import chess
import chess.svg
import random
import concurrent.futures
import time
import IPython
import os
import json
from google.colab import files
from google.colab import output
import ipywidgets as widgets
from IPython.display import display, SVG, clear_output, HTML
from ipywidgets import interact, IntSlider
import copy
import chess.polyglot
from concurrent.futures import ProcessPoolExecutor
import multiprocessing

In [ ]:
# Основні параметри навчання
POPULATION_SIZE = 20
GENERATIONS = 20
MAX_MOVES = 80
MUTATION_RATE = 0.15
TRAINING_TIME_LIMIT = 0.1


# Кешування та пам'ять
evaluation_cache = {}
MAX_CACHE_SIZE = 100000
best_genome_overall = None

Інтелект Бота

In [11]:
class TitanBot:
    """
    Генетичний шаховий бот.
    Використовує NegaMax, Quiescence Search, Таблиці Транспозицій та Bitboards.
    """
    def __init__(self, name="Titan", genome=None, time_limit=1.0):
        self.name = name
        self.time_limit = time_limit
        self.nodes_searched = 0
        self.transposition_table = {}

        # Таблиця цінності клітинок (PST) винесена для швидкості
        self.pst = [
            -5, -4, -3, -3, -3, -3, -4, -5,
            -4, -2,  0,  0,  0,  0, -2, -4,
            -3,  0,  1,  1,  1,  1,  0, -3,
            -3,  0,  1,  2,  2,  1,  0, -3,
            -3,  0,  1,  2,  2,  1,  0, -3,
            -3,  0,  1,  1,  1,  1,  0, -3,
            -4, -2,  0,  0,  0,  0, -2, -4,
            -5, -4, -3, -3, -3, -3, -4, -5
        ]

        if genome is None:
            self.genome = self.generate_random_genome()
        else:
            self.genome = genome.copy()

    @staticmethod
    def generate_random_genome():
        """Словник генів """
        return {
            'pawn': random.uniform(80, 120),
            'knight': random.uniform(280, 320),
            'bishop': random.uniform(290, 330),
            'rook': random.uniform(480, 520),
            'queen': random.uniform(850, 950),
            'center_control': random.uniform(5, 20),
            'pawn_advancement': random.uniform(2, 10),
            'king_safety': random.uniform(10, 30),
            'mobility': random.uniform(0.5, 2.0)
        }

    def evaluate_board(self, board):
        """Оцінка позиції за допомогою Bitboards"""
        if board.is_checkmate():
            # Мат оцінюється найвище. Віднімаємо глибину до мату,
            # але тут просто повертаємо величезне число.
            return -99999 if board.turn else 99999
        if board.is_stalemate() or board.is_insufficient_material():
            return 0

        score = 0

        # 1. Матеріальна перевага (Швидкий підрахунок через bitboards)
        for piece_type in [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN]:
            val = self.genome[self._get_piece_name(piece_type)]
            white_count = len(board.pieces(piece_type, chess.WHITE))
            black_count = len(board.pieces(piece_type, chess.BLACK))
            score += (white_count - black_count) * val

        # 2. Позиційна перевага
        center_squares = {chess.D4, chess.E4, chess.D5, chess.E5}
        white_pawns = board.pieces(chess.PAWN, chess.WHITE)
        black_pawns = board.pieces(chess.PAWN, chess.BLACK)

        # Центр та просування для білих пішаків
        for sq in white_pawns:
            if sq in center_squares: score += self.genome['center_control']
            rank = chess.square_rank(sq)
            if rank > 3: score += (rank - 3) * self.genome['pawn_advancement']
            score += self.pst[sq] * (self.genome['center_control'] * 0.1)

        # Центр та просування для чорних пішаків
        for sq in black_pawns:
            if sq in center_squares: score -= self.genome['center_control']
            rank = chess.square_rank(sq)
            if rank < 4: score -= (4 - rank) * self.genome['pawn_advancement']
            score -= self.pst[chess.square_mirror(sq)] * (self.genome['center_control'] * 0.1)

        # 3. Мобільність
        mobility_score = board.legal_moves.count() * self.genome['mobility']
        score += mobility_score if board.turn == chess.WHITE else -mobility_score

        # Повертаємо оцінку відносно того, чий зараз хід (правило NegaMax)
        return score if board.turn == chess.WHITE else -score

    def _get_piece_name(self, piece_type):
        names = {1: 'pawn', 2: 'knight', 3: 'bishop', 4: 'rook', 5: 'queen'}
        return names.get(piece_type, 'pawn')

    def order_moves(self, board, captures_only=False):
        """MVV-LVA: Слабкий б'є Сильного (Most Valuable Victim - Least Valuable Attacker)"""
        moves = list(board.generate_legal_captures()) if captures_only else list(board.legal_moves)

        def move_score(move):
            score = 0
            if board.is_capture(move):
                victim = board.piece_at(move.to_square)
                attacker = board.piece_at(move.from_square)
                v_val = victim.piece_type if victim else 1
                a_val = attacker.piece_type if attacker else 1
                score = 10 * v_val - a_val + 100 # Захоплення завжди пріоритетні

            if move.promotion: score += 90
            return score

        moves.sort(key=move_score, reverse=True)
        return moves

    def quiescence(self, board, alpha, beta):
        """Форсований пошук: не зупиняємось посеред "взяття" фігур"""
        self.nodes_searched += 1
        stand_pat = self.evaluate_board(board)

        if stand_pat >= beta:
            return beta
        if alpha < stand_pat:
            alpha = stand_pat

        for move in self.order_moves(board, captures_only=True):
            board.push(move)
            score = -self.quiescence(board, -beta, -alpha)
            board.pop()

            if score >= beta:
                return beta
            if score > alpha:
                alpha = score

        return alpha

    def negamax(self, board, depth, alpha, beta, end_time):
        """Основний рушій: NegaMax з Альфа-Бета відсіканням та Таблицею Транспозицій"""
        if time.time() > end_time:
            raise TimeoutError()

        self.nodes_searched += 1

        # Перевірка Таблиці Транспозицій
        hash_key = board.fen()
        if hash_key in self.transposition_table:
            tt_depth, tt_score = self.transposition_table[hash_key]
            if tt_depth >= depth:
                return tt_score

        if depth <= 0 or board.is_game_over():
            return self.quiescence(board, alpha, beta)

        best_score = -float('inf')

        for move in self.order_moves(board):
            board.push(move)
            try:
                score = -self.negamax(board, depth - 1, -beta, -alpha, end_time)
            finally:
                board.pop()

            best_score = max(best_score, score)
            alpha = max(alpha, score)

            if alpha >= beta:
                break # Альфа-Бета відсікання

        # Зберігаємо результат у таблицю
        self.transposition_table[hash_key] = (depth, best_score)
        return best_score

    def choose_move(self, board):
        """Ітеративне заглиблення (Iterative Deepening) з контролем часу"""
        self.nodes_searched = 0
        self.transposition_table.clear() # Очищаємо кеш перед новим ходом

        best_move = None
        start_time = time.time()
        end_time = start_time + self.time_limit

        legal_moves = self.order_moves(board)
        if not legal_moves: return None
        if len(legal_moves) == 1: return legal_moves[0]

        try:
            # Шукаємо від глибини 1 до 20, поки не скінчиться час
            for depth in range(1, 20):
                current_best_move = None
                best_score = -float('inf')
                alpha = -float('inf')
                beta = float('inf')

                # Завжди перевіряємо найкращий хід минулої глибини першим
                if best_move and best_move in legal_moves:
                    legal_moves.remove(best_move)
                    legal_moves.insert(0, best_move)

                for move in legal_moves:
                    board.push(move)
                    try:
                        score = -self.negamax(board, depth - 1, -beta, -alpha, end_time)
                    finally:
                        board.pop()

                    if score > best_score:
                        best_score = score
                        current_best_move = move

                    alpha = max(alpha, score)

                best_move = current_best_move

        except TimeoutError:
            pass # Якщо час вийшов, просто повертаємо те, що встигли знайти на попередній глибині

        # print(f"[{self.name}] Прораховано вузлів: {self.nodes_searched}")
        return best_move if best_move else random.choice(list(board.legal_moves))

Тренування ботів гра проти еталону

In [ ]:
def get_adaptive_weights(generation, total_generations):
    """На початку нічия дає непогані бали, в кінці - майже нічого. Цінується тільки перемога."""
    progress = generation / max(1, total_generations - 1)
    win_score = 3 + (2 * progress)
    draw_score = 1.5 - (2.5 * progress)
    loss_score = 0
    return win_score, draw_score, loss_score

def play_game_silent(genome1, genome2):
    board = chess.Board()
    moves = 0

    bot_white = TitanBot("W", genome=genome1, time_limit=TRAINING_TIME_LIMIT)
    bot_black = TitanBot("B", genome=genome2, time_limit=TRAINING_TIME_LIMIT)

    while not board.is_game_over(claim_draw=True) and moves < MAX_MOVES:
        if board.turn == chess.WHITE:
            move = bot_white.choose_move(board)
        else:
            move = bot_black.choose_move(board)

        if not move: break
        board.push(move)
        moves += 1

    if board.is_checkmate(): return 1 if not board.turn else -1
    return 0

def play_match_benchmark(match_data):
    bot_index, is_white, bot_genome, benchmark_genome = match_data
    if is_white:
        res = play_game_silent(bot_genome, benchmark_genome)
    else:
        res = play_game_silent(benchmark_genome, bot_genome)
    return bot_index, is_white, res

def adaptive_crossover(p1, p2, score1, score2):
    total_score = score1 + score2 + 1e-5
    prob_p1 = score1 / total_score
    child = {}
    for key in p1.keys():
        child[key] = p1[key] if random.random() < prob_p1 else p2[key]
    return child

# ГОЛОВНИЙ ЦИКЛ НАВЧАННЯ

population = [TitanBot.generate_random_genome() for _ in range(POPULATION_SIZE)]

try:
    if not isinstance(best_genome_overall, dict):
        best_genome_overall = None
except NameError:
    best_genome_overall = None

if best_genome_overall is None:
    benchmark_bot = TitanBot.generate_random_genome()
else:
    benchmark_bot = best_genome_overall

for gen in range(GENERATIONS):
    print(f"\n{'='*40}")
    print(f"РОЗРАХУНОК ПОКОЛІННЯ {gen + 1}/{GENERATIONS} (Адаптивний Фітнес)")

    win_w, draw_w, loss_w = get_adaptive_weights(gen, GENERATIONS)
    print(f"Ваги: Перемога={win_w:.2f}, Нічия={draw_w:.2f}")
    print(f"{'='*40}")

    scores = {i: 0 for i in range(POPULATION_SIZE)}

    matches = []
    for i in range(POPULATION_SIZE):
        matches.append((i, True, population[i], benchmark_bot))
        matches.append((i, False, population[i], benchmark_bot))

    with concurrent.futures.ProcessPoolExecutor() as ex:
        for bot_index, is_white, res in ex.map(play_match_benchmark, matches):
            if res == 0:
                scores[bot_index] += draw_w
            elif (is_white and res == 1) or (not is_white and res == -1):
                scores[bot_index] += win_w
            else:
                scores[bot_index] += loss_w

    ranked_indices = sorted(range(POPULATION_SIZE), key=lambda k: scores[k], reverse=True)
    current_best_score = scores[ranked_indices[0]]

    # Оновлюємо еталон - тепер це найкращий бот поточного покоління
    best_genome_overall = population[ranked_indices[0]]
    benchmark_bot = best_genome_overall

    print(f"🏆 Найкращий бал у поколінні: {current_best_score:.2f}")

    new_population = []
    new_population.append(population[ranked_indices[0]]) # Еліта 1
    if POPULATION_SIZE > 1:
        new_population.append(population[ranked_indices[1]]) # Еліта 2

    survivors_indices = ranked_indices[:max(2, POPULATION_SIZE // 2)]

    while len(new_population) < POPULATION_SIZE:
        p1_idx, p2_idx = random.sample(survivors_indices, 2)
        child = adaptive_crossover(population[p1_idx], population[p2_idx], scores[p1_idx], scores[p2_idx])

        # Стабільна мутація без стрибків
        if random.random() < MUTATION_RATE:
            random_key = random.choice(list(child.keys()))
            child[random_key] *= random.uniform(0.8, 1.2)

        new_population.append(child)

    population = new_population

print("\n✨ Навчання завершено!")
print("Геном Чемпіона:")
for k, v in best_genome_overall.items():
    print(f"  {k}: {v:.2f}")


РОЗРАХУНОК ПОКОЛІННЯ 1/20 (Адаптивний Фітнес)
Ваги: Перемога=3.00, Нічия=1.50
🏆 Найкращий бал у поколінні: 3.00

РОЗРАХУНОК ПОКОЛІННЯ 2/20 (Адаптивний Фітнес)
Ваги: Перемога=3.11, Нічия=1.37
🏆 Найкращий бал у поколінні: 4.47

РОЗРАХУНОК ПОКОЛІННЯ 3/20 (Адаптивний Фітнес)
Ваги: Перемога=3.21, Нічия=1.24
🏆 Найкращий бал у поколінні: 3.21

РОЗРАХУНОК ПОКОЛІННЯ 4/20 (Адаптивний Фітнес)
Ваги: Перемога=3.32, Нічия=1.11
🏆 Найкращий бал у поколінні: 4.42

РОЗРАХУНОК ПОКОЛІННЯ 5/20 (Адаптивний Фітнес)
Ваги: Перемога=3.42, Нічия=0.97
🏆 Найкращий бал у поколінні: 4.39

РОЗРАХУНОК ПОКОЛІННЯ 6/20 (Адаптивний Фітнес)
Ваги: Перемога=3.53, Нічия=0.84
🏆 Найкращий бал у поколінні: 1.68

РОЗРАХУНОК ПОКОЛІННЯ 7/20 (Адаптивний Фітнес)
Ваги: Перемога=3.63, Нічия=0.71
🏆 Найкращий бал у поколінні: 4.34

РОЗРАХУНОК ПОКОЛІННЯ 8/20 (Адаптивний Фітнес)
Ваги: Перемога=3.74, Нічия=0.58
🏆 Найкращий бал у поколінні: 4.32

РОЗРАХУНОК ПОКОЛІННЯ 9/20 (Адаптивний Фітнес)
Ваги: Перемога=3.84, Нічия=0.45
🏆 Найкращий бал у

Людина проти бота

In [12]:

# Перевіряємо, чи існує змінна best_genome_overall і чи це словник
try:
    if isinstance(best_genome_overall, dict):
        titan_genome = best_genome_overall
    else:
        titan_genome = None # Якщо це старий список, скидаємо
except NameError:
    titan_genome = None

# Створюємо супротивника! (Час на роздуми: 1.5 секунди)
current_bot = TitanBot(name="Titan", genome=titan_genome, time_limit=1.5)

# Глобальні змінні стану гри
play_board = chess.Board()
player_color = chess.WHITE

def get_status_message(board):
    """Перетворює стан дошки у зрозумілий текст результату"""
    if board.is_checkmate():
        winner = "Білі" if board.outcome().winner == chess.WHITE else "Чорні"
        return f"🏁 МАТ! Перемогли {winner}."
    if board.is_stalemate():
        return "🤝 НІЧИЯ! Пат."
    if board.is_insufficient_material():
        return "🤝 НІЧИЯ! Недостатньо фігур для мату."
    if board.can_claim_draw():
        return "🤝 НІЧИЯ! Повторення ходів."
    if board.is_game_over():
        return f"Гра завершена. Результат: {board.result()}"
    return "Твій хід..."

def handle_move(move_str):
    """Обробник ходів з посиленою перевіркою кінця гри"""
    global play_board, player_color

    try:
        move = chess.Move.from_uci(move_str)
    except:
        return IPython.display.JSON({'fen': play_board.fen(), 'status': '❌ Помилка формату'})

    if move in play_board.legal_moves:
        # 1. Хід гравця
        play_board.push(move)

        if play_board.is_game_over() or play_board.can_claim_draw():
            return IPython.display.JSON({
                'fen': play_board.fen(),
                'status': get_status_message(play_board)
            })

        # 2. Хід TitanBot
        bot_move = current_bot.choose_move(play_board)
        if bot_move:
            play_board.push(bot_move)

        status = get_status_message(play_board)
        return IPython.display.JSON({
            'fen': play_board.fen(),
            'status': status
        })
    else:
        return IPython.display.JSON({
            'fen': play_board.fen(),
            'status': '❌ Нелегальний хід!'
        })

output.register_callback('notebook.handle_move', handle_move)

def start_new_game(b):
    global play_board, player_color
    play_board = chess.Board()

    choice = color_dropdown.value
    if choice == 'Випадково':
        player_color = random.choice([chess.WHITE, chess.BLACK])
    elif choice == 'Білі':
        player_color = chess.WHITE
    else:
        player_color = chess.BLACK

    board_orientation = 'white' if player_color == chess.WHITE else 'black'

    initial_status = "Твій хід..."
    if player_color == chess.BLACK:
        # Перший хід TitanBot
        bot_move = current_bot.choose_move(play_board)
        if bot_move:
            play_board.push(bot_move)
        initial_status = get_status_message(play_board)

    html_code = f"""
    <link rel="stylesheet" href="https://unpkg.com/@chrisoakman/chessboardjs@1.0.0/dist/chessboard-1.0.0.min.css" />
    <script src="https://code.jquery.com/jquery-3.5.1.min.js"></script>
    <script src="https://unpkg.com/@chrisoakman/chessboardjs@1.0.0/dist/chessboard-1.0.0.min.js"></script>

    <div style="display: flex; flex-direction: column; align-items: center; padding: 20px; font-family: sans-serif;">
        <div id="board_div" style="width: 400px; margin: 10px 0; box-shadow: 0px 4px 10px rgba(0,0,0,0.3);"></div>
        <h3 id="statusText" style="margin-top: 15px; color: #1a73e8; min-height: 24px;">{initial_status}</h3>
    </div>

    <script>
    setTimeout(function() {{
        var config = {{
          draggable: true,
          position: '{play_board.fen()}',
          orientation: '{board_orientation}',
          pieceTheme: 'https://chessboardjs.com/img/chesspieces/wikipedia/{{piece}}.png',

          onDragStart: function(source, piece, position, orientation) {{
              if (document.getElementById('statusText').innerText.includes('🏁') ||
                  document.getElementById('statusText').innerText.includes('🤝')) return false;
              if ((orientation === 'white' && piece.search(/^b/) !== -1) ||
                  (orientation === 'black' && piece.search(/^w/) !== -1)) return false;
          }},

          onDrop: async function(source, target, piece, newPos, oldPos, orientation) {{
              var move = source + target;
              if (piece === 'wP' && target[1] === '8') move += 'q';
              if (piece === 'bP' && target[1] === '1') move += 'q';

              document.getElementById('statusText').innerText = "⏳ TitanBot думає...";

              try {{
                  const result = await google.colab.kernel.invokeFunction('notebook.handle_move', [move], {{}});
                  const data = result.data['application/json'];
                  window.myBoard.position(data.fen);
                  document.getElementById('statusText').innerText = data.status;
              }} catch(e) {{
                  document.getElementById('statusText').innerText = "❌ Помилка з'єднання.";
              }}
          }}
        }};
        window.myBoard = Chessboard('board_div', config);
    }}, 500);
    </script>
    """
    with output_area:
        IPython.display.clear_output(wait=True)
        IPython.display.display(IPython.display.HTML(html_code))
# ВІДМАЛЬОВКА ІНТЕРФЕЙСУ

color_dropdown = widgets.Dropdown(options=['Білі', 'Чорні', 'Випадково'], value='Білі', description='Колір:')
start_btn = widgets.Button(description='Грати проти Titan', button_style='success')
start_btn.on_click(start_new_game)
output_area = widgets.Output()

display(widgets.HBox([color_dropdown, start_btn]))
display(output_area)
start_new_game(None)

Output()

Збереження бота

In [ ]:
import json
from google.colab import files

# Перевіряємо, чи є кого зберігати
if best_genome_overall is not None and isinstance(best_genome_overall, dict):
    filename = 'titan_champion.json'

    # Записуємо словник генів у файл формату JSON
    # indent=4 робить файл відформатованим (стовпчиком)
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(best_genome_overall, f, indent=4)

    print(f" Геном успішно записано у файл {filename}.")
    print("Запускаю завантаження на твій пристрій...")

    # Викликаємо вікно завантаження браузера
    files.download(filename)
else:
    print(" Немає збереженого чемпіона нового формату! Спочатку запусти Блок 4.")

 Геном успішно записано у файл titan_champion.json.
Запускаю завантаження на твій пристрій...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Завантаження бота

In [1]:
import json
from google.colab import files

print("Виберіть файл збереженого бота (напр. titan_champion.json) з вашого комп'ютера:")

# Відкриває діалогове вікно для вибору файлу
uploaded = files.upload()

if uploaded:
    # Беремо ім'я першого завантаженого файлу
    filename = list(uploaded.keys())[0]

    # Зчитуємо дані з файлу
    with open(filename, 'r', encoding='utf-8') as f:
        best_genome_overall = json.load(f)

    print("\n Бота успішно завантажено і встановлено як поточного Чемпіона!")
    print("Його геном:")

    # Красиво виводимо словник
    for key, value in best_genome_overall.items():
        print(f"  {key}: {value:.2f}")

    print("\nТепер ти можеш одразу переходити до Блоку 6 і грати проти нього!")
else:
    print(" Файл не було завантажено.")

Виберіть файл збереженого бота (напр. titan_champion.json) з вашого комп'ютера:


Saving titan_champion (1).json to titan_champion (1).json

 Бота успішно завантажено і встановлено як поточного Чемпіона!
Його геном:
  pawn: 69.72
  knight: 309.89
  bishop: 323.88
  rook: 514.00
  queen: 852.63
  center_control: 7.26
  pawn_advancement: 9.39
  king_safety: 17.15
  mobility: 1.64

Тепер ти можеш одразу переходити до Блоку 6 і грати проти нього!


[1.429, 3.954, 3.045, 4.134, 7.278, 0.267, 1.212, 0.012, 0.382, 0.76, 0.768]

[0.941, 3.206, 3.934, 4.174, 9.113, 0.415, 0.553, 0.041, 0.263, 0.375, 0.101]